In [1]:
import polars as pl

places = pl.read_json("pleiades-places-latest.json").select("@graph").explode("@graph").unnest("@graph")

In [2]:
from pprint import pprint
# only places that are of settlement type 
settlements = places.filter(pl.col("placeTypes").list.contains("settlement"))

# only places that have at least 1 location associated with them
settlements = settlements.filter(pl.col("locations").list.len() > 0)

# use start and end dates of Pleiades' location data as a proxy for inhabitation
settlements = settlements.with_columns(
    inhabitation_start=pl.col("locations").map_elements(
        lambda locs: min([loc["start"] for loc in locs if loc["start"] is not None], default=None)
    ),
    inhabitation_end=pl.col("locations").map_elements(
        lambda locs: max([loc["end"] for loc in locs if loc["end"] is not None], default=None)
    )
)

# remove settlements without inhabitation_start
settlements = settlements.filter(pl.col("inhabitation_start").is_not_null())

# remove settlements that started after 1000BC
settlements = settlements.filter(pl.col("inhabitation_start") <= -1000)

In [4]:
output = settlements.with_columns([
    pl.col("title").alias("name"), 
    pl.struct(
        longitude=pl.col("reprPoint").list.get(0),
        latitude=pl.col("reprPoint").list.get(1)
    ).alias("location"),
    pl.struct(
        start=pl.col("inhabitation_start"),
        end=pl.col("inhabitation_end")
    ).alias("inhabitation"),
    pl.col("references").list.eval(
        pl.when(pl.element().struct.field("accessURI").str.starts_with("https://en.wikipedia.org"))
        .then(pl.element().struct.field("accessURI"))
    ).list.drop_nulls()
    .list.first()
    .alias("wikipediaURL"),
    pl.col("uri").alias("pleiadesURI")
])["name", "wikipediaURL", "location", "inhabitation", "pleiadesURI", "description"]

output.write_json("settlements-pleiades.json")